# Editing databases for Wagtail (v1.2.1)

Helper notebook for reshaping taxonomy and reference databases into a format compatible with **Wagtail v1.2.1**.

Tested on the four databases published at <https://doi.org/10.5281/zenodo.21817761>.

> **Note:** some databases need additional manual curation. Use `get_problem_rows()` to inspect anything the automatic fixes cannot resolve.

**Workflow:** `load_taxonomy` → `get_problem_rows` (inspect) → apply fixes → `get_problem_rows` (re-check) → `write_taxonomy`

## Expected output format

### `(marker)_taxonomy*` — tab-separated

```text
feature_id      taxonomy
FLASV1.1346;tax=d:Bacteria,p:Actinobacteriota,c:Actinobacteria,o:Propionibacteriales,f:Propionibacteriaceae,g:Cutibacterium,s:Cutibacterium_acnes;      d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Propionibacteriales;f__Propionibacteriaceae;g__Cutibacterium;s__Cutibacterium_acnes
FLASV2.1310;tax=d:Bacteria,p:Proteobacteria,c:Alphaproteobacteria,o:Rhizobiales,f:Xanthobacteraceae,g:Bradyrhizobium,s:MFD_s_2; d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhizobiales;f__Xanthobacteraceae;g__Bradyrhizobium;s__MFD_s_2
```

### `(marker)_database*` — FASTA

```text
>FLASV1.1346;tax=d:Bacteria,p:Actinobacteriota,c:Actinobacteria,o:Propionibacteriales,f:Propionibacteriaceae,g:Cutibacterium,s:Cutibacterium_acnes;
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAA
>FLASV2.1310;tax=d:Bacteria,p:Proteobacteria,c:Alphaproteobacteria,o:Rhizobiales,f:Xanthobacteraceae,g:Bradyrhizobium,s:MFD_s_2;
AGCGAACGCTGGCGGCAGGCTTAACAC
```

## Setup

Requires `polars` (`pip install polars`).

In [ ]:
import re
from collections import Counter

import polars as pl

# --- taxonomy level parsing --------------------------------------------------

# a single level token, e.g. 'g__Bacillus' -> ('g', 'Bacillus')
LEVEL_RE = re.compile(r"^([a-z]+)__(.*)$")

# accepted root ranks: d__ (domain) or k__ (kingdom)
ROOT_PREFIXES = {"d", "k"}

# ranks allowed to appear anywhere without breaking the expected order
# (e.g. UNITE species hypotheses)
OPTIONAL_EXTRA_RANKS = {"sh"}

# canonical rank orders
EXPECTED_AFTER_D = ["d", "p", "c", "o", "f", "g", "s"]
EXPECTED_AFTER_K = ["k", "p", "c", "o", "f", "g", "s"]

# PR2 rank order: Domain, Supergroup, Division, Subdivision,
#                 Class, Order, Family, Genus, Species
PR2_PREFIXES = ["d", "sg", "p", "sp", "c", "o", "f", "g", "s"]

# every rank order get_problem_rows() considers valid
KNOWN_SCHEMES = [EXPECTED_AFTER_D, EXPECTED_AFTER_K, PR2_PREFIXES]

## Loading

In [ ]:
def load_taxonomy(path: str) -> pl.DataFrame:
    """
    Load a taxonomy TSV into a two-column DataFrame (feature_id, taxonomy).

    Handles optional header rows: 'Feature ID', 'feature_id', '#OTUid'.
    """
    raw = pl.read_csv(
        path,
        separator="\t",
        has_header=False,
        comment_prefix="#",
        new_columns=["feature_id", "taxonomy"],
        infer_schema_length=0,
        truncate_ragged_lines=True,
    ).select(["feature_id", "taxonomy"])

    header_values = {"feature id", "feature_id", "#otuid", "otu id", "taxon"}
    if raw.height and raw["feature_id"][0].strip().lower() in header_values:
        raw = raw.slice(1)

    df = raw.with_columns(
        pl.col("feature_id").str.strip_chars(),
        pl.col("taxonomy").str.strip_chars(),
    )

    print(f"load_taxonomy: {df.height} rows loaded from {path}")
    return df

## Inspecting problems

Run this **before** applying fixes to see what needs attention, and again afterwards to confirm nothing is left.

Detected issues:

| issue | meaning |
| --- | --- |
| `trailing_semicolon` | taxonomy ends with `;` |
| `inter_level_spaces` | space after a `;` separator |
| `whitespace_in_levels` | space inside a level name, e.g. `s__Clydonella sawyeri` |
| `empty_level` | empty token between separators, e.g. `;;` |
| `missing_prefix` | a level without an `x__` prefix |
| `unexpected_root_rank` | first level is not `d__` or `k__` |
| `unexpected_rank_order` | ranks do not follow a known scheme |

In [ ]:
def _is_subsequence(prefixes: list, scheme: list) -> bool:
    """True if `prefixes` appears in `scheme` in order (gaps allowed)."""
    it = iter(scheme)
    return all(p in it for p in prefixes)


def _row_issues(tax: str) -> list:
    """Return the list of structural issues found in one taxonomy string."""
    issues = []

    if re.search(r";+\s*$", tax):
        issues.append("trailing_semicolon")
    if re.search(r";\s+", tax):
        issues.append("inter_level_spaces")

    tokens = [t.strip() for t in tax.rstrip(";").split(";")]
    if any(not t for t in tokens):
        issues.append("empty_level")

    prefixes = []
    for tok in tokens:
        if not tok:
            continue
        m = LEVEL_RE.match(tok)
        if not m:
            issues.append("missing_prefix")
            continue
        prefixes.append(m.group(1))
        if " " in m.group(2):
            issues.append("whitespace_in_levels")

    if prefixes:
        if prefixes[0] not in ROOT_PREFIXES:
            issues.append("unexpected_root_rank")
        ranked = [p for p in prefixes if p not in OPTIONAL_EXTRA_RANKS]
        if not any(_is_subsequence(ranked, s) for s in KNOWN_SCHEMES):
            issues.append("unexpected_rank_order")

    return sorted(set(issues))


def get_problem_rows(
    df: pl.DataFrame,
    issue: str = None,
    col: str = "taxonomy",
) -> pl.DataFrame:
    """
    Return rows whose taxonomy has structural problems, with an `issues` column.

    Pass `issue` to filter to a single problem type, e.g.
        get_problem_rows(df, "whitespace_in_levels")
    """
    issues = [";".join(_row_issues(t)) for t in df[col].to_list()]

    out = (
        df
        .with_columns(pl.Series("issues", issues))
        .filter(pl.col("issues") != "")
    )

    counts = Counter(i for row in out["issues"].to_list() for i in row.split(";"))
    print(f"get_problem_rows: {out.height} of {df.height} rows have issues")
    for name, count in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  {name}: {count}")

    if issue:
        out = out.filter(pl.col("issues").str.contains(issue))
        print(f"  filtered to '{issue}': {out.height} rows")

    return out

## Cleaning

Each function returns a new DataFrame and prints how many rows it changed, so they can be chained safely.

In [ ]:
def _apply_fix(
    df: pl.DataFrame,
    fixed: pl.Series,
    label: str,
    col: str = "taxonomy",
) -> pl.DataFrame:
    """Replace `col` with `fixed`, reporting how many rows changed."""
    changed = int((fixed != df[col]).sum())
    print(f"{label}: {changed} rows updated")
    return df.with_columns(fixed.alias(col))


def fix_trailing_semicolon(df: pl.DataFrame) -> pl.DataFrame:
    """Remove trailing semicolons (and trailing whitespace) from taxonomy."""
    fixed = df["taxonomy"].str.replace(r";+\s*$", "").str.strip_chars()
    return _apply_fix(df, fixed, "fix_trailing_semicolon")


def strip_inter_level_spaces(df: pl.DataFrame) -> pl.DataFrame:
    """
    Remove spaces after semicolons (between levels).

    'd__Bacteria; p__Firmicutes' -> 'd__Bacteria;p__Firmicutes'
    """
    fixed = df["taxonomy"].str.replace_all(r";\s+", ";")
    return _apply_fix(df, fixed, "strip_inter_level_spaces")


def fix_whitespace_in_names(df: pl.DataFrame, replacement: str = "_") -> pl.DataFrame:
    """
    Replace spaces inside level names with `replacement` (default '_').

    's__Clydonella sawyeri_2201168' -> 's__Clydonella_sawyeri_2201168'

    Review ambiguous cases first with
        get_problem_rows(df, "whitespace_in_levels")
    """
    def _fix(tax: str) -> str:
        out = []
        for level in tax.split(";"):
            token = level.strip()
            if not token:
                continue
            m = LEVEL_RE.match(token)
            if m:
                out.append(f"{m.group(1)}__{m.group(2).replace(' ', replacement)}")
            else:
                out.append(token.replace(" ", replacement))
        return ";".join(out)

    fixed = pl.Series([_fix(t) for t in df["taxonomy"].to_list()])
    return _apply_fix(df, fixed, "fix_whitespace_in_names")

## Adding rank prefixes

Some databases ship taxonomy strings with no `x__` rank markers at all. `add_rank_prefixes` applies a scheme by position, so the level count has to match:

| scheme | prefixes | levels |
| --- | --- | --- |
| `EXPECTED_AFTER_D` (default) | `d p c o f g s` | 7 |
| `EXPECTED_AFTER_K` | `k p c o f g s` | 7 |
| `PR2_PREFIXES` | `d sg p sp c o f g s` | 9 |

Rows that already carry prefixes pass through untouched, and rows with the wrong level count are skipped with a warning rather than silently mangled — check those with `get_problem_rows(df, "missing_prefix")`.

`add_pr2_prefixes` is a thin wrapper for the PR2 scheme.

In [ ]:
def add_rank_prefixes(
    df: pl.DataFrame,
    scheme: list = None,
    col: str = "taxonomy",
) -> pl.DataFrame:
    """
    Add rank prefixes to plain taxonomy strings that carry no `x__` markers.

    `scheme` is the ordered list of prefixes to apply; each row must have
    exactly as many levels as the scheme is long. Defaults to EXPECTED_AFTER_D.

        EXPECTED_AFTER_D   d p c o f g s        domain-rooted,  7 levels
        EXPECTED_AFTER_K   k p c o f g s        kingdom-rooted, 7 levels
        PR2_PREFIXES       d sg p sp c o f g s  PR2,            9 levels

    Rows that already have prefixes are left untouched.
    Trailing semicolons are stripped first.
    Rows whose level count differs from the scheme are skipped with a warning.

    Example:
        df = add_rank_prefixes(df)                    # d__ ... s__
        df = add_rank_prefixes(df, EXPECTED_AFTER_K)  # k__ ... s__
    """
    if scheme is None:
        scheme = EXPECTED_AFTER_D

    skipped: Counter = Counter()

    def _add(tax: str) -> str:
        # already prefixed?
        if LEVEL_RE.match(tax.split(";")[0].strip()):
            return tax

        levels = [t.strip() for t in tax.rstrip(";").split(";") if t.strip()]
        if len(levels) != len(scheme):
            skipped[len(levels)] += 1
            return tax  # unchanged - reported in the warning below

        return ";".join(f"{pre}__{name}" for pre, name in zip(scheme, levels))

    fixed = pl.Series([_add(t) for t in df[col].to_list()])
    df = _apply_fix(df, fixed, f"add_rank_prefixes[{len(scheme)} ranks]", col=col)

    for n_levels, count in sorted(skipped.items()):
        print(f"  WARNING: {count} row(s) skipped - expected {len(scheme)} levels, got {n_levels}")

    return df

In [ ]:
def add_pr2_prefixes(df: pl.DataFrame, col: str = "taxonomy") -> pl.DataFrame:
    """
    Add PR2 rank prefixes to plain PR2-style taxonomy strings (9 levels).

    PR2 rank order:
        d__   Domain
        sg__  Supergroup
        p__   Division      (phylum equivalent)
        sp__  Subdivision
        c__   Class
        o__   Order
        f__   Family
        g__   Genus
        s__   Species

    Convenience wrapper around add_rank_prefixes(df, PR2_PREFIXES).

    Example:
        df = add_pr2_prefixes(df)
    """
    return add_rank_prefixes(df, PR2_PREFIXES, col=col)

## Writing output

The output keeps the `feature_id` / `taxonomy` header row, matching the input format. Pass `include_header=False` if you ever need a headerless file.

In [ ]:
def write_taxonomy(
    df: pl.DataFrame,
    path: str,
    include_header: bool = True,
) -> None:
    """Write feature_id + taxonomy as a TSV (header row included by default)."""
    (
        df
        .select(["feature_id", "taxonomy"])
        .write_csv(path, separator="\t", include_header=include_header)
    )
    print(f"write_taxonomy: {df.height} rows written to {path}")

## Example run

Edit the two paths below, then run the cells. The fixes you need depend on the database — inspect first, then apply only what `get_problem_rows` reports.

In [ ]:
INPUT_TAXONOMY = "path/to/your_taxonomy.tsv"            # <- edit me
OUTPUT_TAXONOMY = "path/to/your_taxonomy_wagtail.tsv"   # <- edit me

df = load_taxonomy(INPUT_TAXONOMY)
df.head()

In [ ]:
# inspect before changing anything
problems = get_problem_rows(df)
problems.head()

In [ ]:
# apply only the fixes the report above calls for
# df = fix_trailing_semicolon(df)
# df = strip_inter_level_spaces(df)
df = fix_whitespace_in_names(df)                 # check whitespace_in_levels rows first
# df = add_rank_prefixes(df)                       # unprefixed, 7 levels, d__ root
# df = add_rank_prefixes(df, EXPECTED_AFTER_K)     # unprefixed, 7 levels, k__ root
# df = add_pr2_prefixes(df)                        # unprefixed, 9 levels, PR2

# confirm the issues are resolved
_ = get_problem_rows(df)

In [ ]:
write_taxonomy(df, OUTPUT_TAXONOMY)